## 1. Imports et Configuration

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from backend.app.etl import load_and_preprocess, get_dataset_statistics
from backend.app.predict import predict_risk
from backend.app.config import RAW_DATA_PATH, PROCESSED_DATA_PATH

# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 2. Chargement des Données

In [ ]:
# Charger le dataset brut
df_raw = pd.read_csv(RAW_DATA_PATH)

print(f"Dataset brut : {df_raw.shape[0]} lignes, {df_raw.shape[1]} colonnes")
df_raw.head()

## 3. Statistiques Descriptives

In [ ]:
# Statistiques de base
df_raw.describe()

In [ ]:
# Informations sur les colonnes
df_raw.info()

In [ ]:
# Valeurs manquantes
missing = df_raw.isnull().sum()
print("Valeurs manquantes par colonne :")
print(missing[missing > 0])

## 4. Visualisations

In [ ]:
# Distribution de l'âge
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(df_raw['age'], bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Âge')
plt.ylabel('Fréquence')
plt.title('Distribution de l\'Âge')

plt.subplot(1, 2, 2)
sns.boxplot(x='has_disease', y='age', data=df_raw)
plt.xlabel('Maladie (0=Sain, 1=Malade)')
plt.ylabel('Âge')
plt.title('Âge selon le Statut de Santé')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution de l'IMC
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(df_raw['bmi'], bins=30, edgecolor='black', alpha=0.7, color='coral')
plt.xlabel('IMC')
plt.ylabel('Fréquence')
plt.title('Distribution de l\'IMC')

plt.subplot(1, 2, 2)
sns.boxplot(x='has_disease', y='bmi', data=df_raw)
plt.xlabel('Maladie (0=Sain, 1=Malade)')
plt.ylabel('IMC')
plt.title('IMC selon le Statut de Santé')

plt.tight_layout()
plt.show()

In [ ]:
# Taux de maladie
plt.figure(figsize=(8, 6))
disease_counts = df_raw['has_disease'].value_counts()
plt.pie(disease_counts, labels=['Sain', 'Malade'], autopct='%1.1f%%', startangle=90)
plt.title('Répartition Sain / Malade')
plt.show()

print(f"Taux de maladie : {df_raw['has_disease'].mean():.2%}")

In [ ]:
# Facteurs de risque
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sexe
sns.countplot(x='sex', hue='has_disease', data=df_raw, ax=axes[0, 0])
axes[0, 0].set_title('Maladie par Sexe')

# Fumeur
sns.countplot(x='smoker', hue='has_disease', data=df_raw, ax=axes[0, 1])
axes[0, 1].set_title('Maladie par Statut Fumeur')

# Activité physique
sns.countplot(x='physical_activity_level', hue='has_disease', data=df_raw, ax=axes[1, 0])
axes[1, 0].set_title('Maladie par Niveau d\'Activité')
axes[1, 0].tick_params(axis='x', rotation=45)

# Hypertension
sns.countplot(x='hypertension', hue='has_disease', data=df_raw, ax=axes[1, 1])
axes[1, 1].set_title('Maladie par Hypertension')

plt.tight_layout()
plt.show()

## 5. Corrélations

In [ ]:
# Encodage pour la matrice de corrélation
df_encoded = df_raw.copy()
df_encoded['sex'] = df_encoded['sex'].map({'M': 1, 'F': 0})
df_encoded['smoker'] = df_encoded['smoker'].map({'yes': 1, 'no': 0})
df_encoded['cholesterol_level'] = df_encoded['cholesterol_level'].map({'high': 1, 'normal': 0})
df_encoded['physical_activity_level'] = df_encoded['physical_activity_level'].map({'low': 0, 'moderate': 1, 'high': 2})

# Matrice de corrélation
plt.figure(figsize=(12, 10))
correlation_matrix = df_encoded.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matrice de Corrélation')
plt.tight_layout()
plt.show()

## 6. Test du Modèle de Prédiction

In [ ]:
# Test avec un patient à haut risque
patient_high_risk = {
    'age': 65,
    'sex': 'M',
    'bmi': 32.5,
    'smoker': 'yes',
    'physical_activity_level': 'low',
    'hypertension': 1,
    'cholesterol_level': 'high',
    'family_history': 1
}

result = predict_risk(patient_high_risk)
print("Patient à HAUT RISQUE :")
print(f"  Score de risque : {result['risk_score']:.2%}")
print(f"  Niveau de risque : {result['risk_level']}")
print(f"  Confiance : {result['confidence']:.2%}")

In [ ]:
# Test avec un patient à faible risque
patient_low_risk = {
    'age': 25,
    'sex': 'F',
    'bmi': 22.0,
    'smoker': 'no',
    'physical_activity_level': 'high',
    'hypertension': 0,
    'cholesterol_level': 'normal',
    'family_history': 0
}

result = predict_risk(patient_low_risk)
print("Patient à FAIBLE RISQUE :")
print(f"  Score de risque : {result['risk_score']:.2%}")
print(f"  Niveau de risque : {result['risk_level']}")
print(f"  Confiance : {result['confidence']:.2%}")

## 7. Statistiques du Dataset

In [ ]:
# Récupérer les statistiques via l'API
stats = get_dataset_statistics()

print("=" * 60)
print("STATISTIQUES DU DATASET")
print("=" * 60)
print(f"\nTotal d'échantillons : {stats['total_samples']}")
print(f"Taux de maladie : {stats['disease_rate']:.2%}")
print(f"\nÂge moyen : {stats['age_distribution']['mean']:.1f} ans")
print(f"IMC moyen : {stats['bmi_distribution']['mean']:.1f}")
print(f"\nTaux de fumeurs : {stats['smoker_rate']:.2%}")
print(f"Taux d'hypertension : {stats['hypertension_rate']:.2%}")

## 📝 Conclusions

À compléter avec vos observations :

1. **Distribution des données** : ...
2. **Corrélations observées** : ...
3. **Performance du modèle** : ...
4. **Pistes d'amélioration** : ...